# Session 18 — Cloud-Based MLOps Pipeline using Google Cloud Vertex AI

**Goal:** a hands-on look at the **Vertex AI Pipelines SDK** itself (`kfp`) — writing,
compiling, and locally-inspecting a pipeline definition — complementing Session 12's
full deploy-gated pipeline with a closer look at the SDK mechanics: caching,
parameterization, and artifact lineage.

## How this differs from Session 12

Session 12 built one specific pipeline (preprocess → train → gate → deploy) for the
heart disease use case. This session focuses on the **SDK itself** — how KFP v2's
component/pipeline abstractions work, so you can design your own pipeline shapes for
different problems, and inspect what gets compiled without needing a live GCP project.

## Prerequisites

The compile step (Step 1-3) runs fully locally with just `kfp` installed — no GCP
account needed to see the compiled pipeline JSON. Only Step 4 (actually submitting to
Vertex AI) needs a real GCP project.

```bash
pip install kfp
```

In [ ]:
from kfp import dsl
from kfp.dsl import component, Input, Output, Dataset, Model, Metrics
from kfp import compiler

print("kfp SDK ready -- everything below through Step 3 runs locally.")

## Step 1 — Components with explicit typed artifacts

`Input[Dataset]` / `Output[Model]` etc. aren't just documentation — KFP uses them to
build the dependency graph and to track artifact lineage (which exact training run
produced which exact model file) automatically.

In [ ]:
@component(base_image="python:3.11", packages_to_install=["pandas==2.2.0", "scikit-learn==1.4.0"])
def generate_synthetic_data(n_rows: int, output_dataset: Output[Dataset]):
    import pandas as pd
    import numpy as np
    rng = np.random.default_rng(0)
    df = pd.DataFrame({
        "x1": rng.normal(size=n_rows),
        "x2": rng.normal(size=n_rows),
        "target": rng.integers(0, 2, size=n_rows),
    })
    df.to_csv(output_dataset.path, index=False)


@component(base_image="python:3.11", packages_to_install=["pandas==2.2.0", "scikit-learn==1.4.0", "joblib"])
def train_model(dataset: Input[Dataset], n_estimators: int, model: Output[Model], metrics: Output[Metrics]):
    import pandas as pd, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_score

    df = pd.read_csv(dataset.path)
    X, y = df.drop(columns="target"), df["target"]
    clf = RandomForestClassifier(n_estimators=n_estimators, random_state=0)
    cv_score = cross_val_score(clf, X, y, cv=3).mean()
    clf.fit(X, y)

    metrics.log_metric("cv_accuracy", cv_score)
    joblib.dump(clf, model.path)

print("Components defined: generate_synthetic_data, train_model")

## Step 2 — Parameterized pipeline with caching

`@dsl.pipeline` parameters (`n_rows`, `n_estimators`) let you re-run the same
pipeline shape with different settings without editing code — the same idea as
MLflow's logged hyperparameters (Session 1), but for entire pipeline runs. KFP caches
each component's output by its inputs' hash, so re-running with identical parameters
skips recomputation.

In [ ]:
@dsl.pipeline(name="session18-demo-pipeline")
def demo_pipeline(n_rows: int = 1000, n_estimators: int = 100):
    data_task = generate_synthetic_data(n_rows=n_rows)
    train_task = train_model(dataset=data_task.outputs["output_dataset"], n_estimators=n_estimators)
    train_task.set_caching_options(True)

print("Pipeline defined.")

## Step 3 — Compile it and inspect the result

Compiling produces a portable pipeline definition (IR YAML/JSON) — this step needs
no GCP project at all, and the output is worth reading once to see what the SDK's
decorators actually produce under the hood.

In [ ]:
compiler.Compiler().compile(demo_pipeline, "session18_pipeline.yaml")

with open("session18_pipeline.yaml") as f:
    content = f.read()
print(f"Compiled pipeline definition: {len(content)} characters")
print(content[:600])
print("...")

## Step 4 — Submit to Vertex AI (needs a real GCP project)

In [ ]:
PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"
PIPELINE_ROOT = "gs://your-bucket/pipeline-root"

from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=REGION)
job = aiplatform.PipelineJob(
    display_name="session18-demo-run",
    template_path="session18_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={"n_rows": 5000, "n_estimators": 300},
    enable_caching=True,
)
job.submit()
print(f"Pipeline submitted: {job.resource_name}")
print(f"View progress: https://console.cloud.google.com/vertex-ai/pipelines/runs?project={PROJECT_ID}")

## Step 5 — Artifact lineage: tracing a model back to its inputs

Once run, Vertex AI's **ML Metadata** store lets you ask "what data produced this
model?" — the pipeline-level analogue of Session 1's `runs:/<run_id>/model` URIs,
but tracking the whole upstream graph, not just one run.

In [ ]:
lineage_query = '''\
from google.cloud import aiplatform

model_artifact = aiplatform.Artifact.get(resource_id="your-model-artifact-id")
lineage = model_artifact.get_source_artifacts()  # walks upstream: which dataset, which run
for upstream in lineage:
    print(upstream.display_name, upstream.uri)
'''
print(lineage_query)

## What to try next

* Add a third component (`evaluate_model`) that only proceeds to a hypothetical
  `deploy_model` step under a `dsl.If` condition, recreating Session 12's quality
  gate but now understanding exactly how `dsl.If` compiles.
* Change `n_estimators` and re-submit — watch in the Vertex AI console how KFP's
  caching skips `generate_synthetic_data` (since `n_rows` didn't change) but reruns
  `train_model`.